In [10]:
from pyspark.sql import SparkSession, functions as F, types as T
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import split
from dotenv import load_dotenv
from pyspark.sql.window import Window

In [11]:
load_dotenv()

True

In [12]:
spark = SparkSession.builder \
    .appName('pyspark-run-with-gcp-bucket') \
    .config("spark.jars", "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

# Set GCS credentials if necessary
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile", os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))


26/04/06 22:23:33 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [13]:
bucket_name = os.getenv("GCS_BUCKET_NAME")
RAW_ROOT = f"gs://{bucket_name}/raw/movie_details"
OUT_ROOT = f"gs://{bucket_name}/silver"

In [14]:
movie_schema = T.StructType([
    T.StructField("adult", T.BooleanType(), True),
    T.StructField("backdrop_path", T.StringType(), True),
    T.StructField("belongs_to_collection", T.StringType(), True),  # can refine later if needed
    T.StructField("budget", T.LongType(), True),

    T.StructField("genres", T.ArrayType(
        T.StructType([
            T.StructField("id", T.LongType(), True),
            T.StructField("name", T.StringType(), True),
        ])
    ), True),

    T.StructField("homepage", T.StringType(), True),
    T.StructField("id", T.LongType(), True),
    T.StructField("imdb_id", T.StringType(), True),

    T.StructField("origin_country", T.ArrayType(T.StringType()), True),

    T.StructField("original_language", T.StringType(), True),
    T.StructField("original_title", T.StringType(), True),
    T.StructField("overview", T.StringType(), True),
    T.StructField("popularity", T.DoubleType(), True),
    T.StructField("poster_path", T.StringType(), True),

    T.StructField("production_companies", T.ArrayType(
        T.StructType([
            T.StructField("id", T.LongType(), True),
            T.StructField("logo_path", T.StringType(), True),
            T.StructField("name", T.StringType(), True),
            T.StructField("origin_country", T.StringType(), True),
        ])
    ), True),

    T.StructField("production_countries", T.ArrayType(
        T.StructType([
            T.StructField("iso_3166_1", T.StringType(), True),
            T.StructField("name", T.StringType(), True),
        ])
    ), True),

    T.StructField("release_date", T.StringType(), True),
    T.StructField("revenue", T.LongType(), True),
    T.StructField("runtime", T.LongType(), True),

    T.StructField("spoken_languages", T.ArrayType(
        T.StructType([
            T.StructField("english_name", T.StringType(), True),
            T.StructField("iso_639_1", T.StringType(), True),
            T.StructField("name", T.StringType(), True),
        ])
    ), True),

    T.StructField("status", T.StringType(), True),
    T.StructField("tagline", T.StringType(), True),
    T.StructField("title", T.StringType(), True),
    T.StructField("video", T.BooleanType(), True),
    T.StructField("vote_average", T.DoubleType(), True),
    T.StructField("vote_count", T.LongType(), True),

    T.StructField("credits", T.StructType([
        T.StructField("cast", T.ArrayType(
            T.StructType([
                T.StructField("adult", T.BooleanType(), True),
                T.StructField("cast_id", T.LongType(), True),
                T.StructField("character", T.StringType(), True),
                T.StructField("credit_id", T.StringType(), True),
                T.StructField("gender", T.LongType(), True),
                T.StructField("id", T.LongType(), True),
                T.StructField("known_for_department", T.StringType(), True),
                T.StructField("name", T.StringType(), True),
                T.StructField("order", T.LongType(), True),
                T.StructField("original_name", T.StringType(), True),
                T.StructField("popularity", T.DoubleType(), True),
                T.StructField("profile_path", T.StringType(), True),
            ])
        ), True),

        T.StructField("crew", T.ArrayType(
            T.StructType([
                T.StructField("adult", T.BooleanType(), True),
                T.StructField("credit_id", T.StringType(), True),
                T.StructField("department", T.StringType(), True),
                T.StructField("gender", T.LongType(), True),
                T.StructField("id", T.LongType(), True),
                T.StructField("job", T.StringType(), True),
                T.StructField("known_for_department", T.StringType(), True),
                T.StructField("name", T.StringType(), True),
                T.StructField("original_name", T.StringType(), True),
                T.StructField("popularity", T.DoubleType(), True),
                T.StructField("profile_path", T.StringType(), True),
            ])
        ), True),
    ]), True),

    T.StructField("keywords", T.StructType([
        T.StructField("keywords", T.ArrayType(
            T.StructType([
                T.StructField("id", T.LongType(), True),
                T.StructField("name", T.StringType(), True),
            ])
        ), True)
    ]), True),

    T.StructField("release_dates", T.StructType([
        T.StructField("results", T.ArrayType(
            T.StructType([
                T.StructField("iso_3166_1", T.StringType(), True),
                T.StructField("release_dates", T.ArrayType(
                    T.StructType([
                        T.StructField("certification", T.StringType(), True),
                        T.StructField("descriptors", T.ArrayType(T.StringType()), True),
                        T.StructField("iso_639_1", T.StringType(), True),
                        T.StructField("note", T.StringType(), True),
                        T.StructField("release_date", T.StringType(), True),
                        T.StructField("type", T.LongType(), True),
                    ])
                ), True)
            ])
        ), True)
    ]), True),

    T.StructField("videos", T.StructType([
        T.StructField("results", T.ArrayType(
            T.StructType([
                T.StructField("id", T.StringType(), True),
                T.StructField("iso_639_1", T.StringType(), True),
                T.StructField("iso_3166_1", T.StringType(), True),
                T.StructField("key", T.StringType(), True),
                T.StructField("name", T.StringType(), True),
                T.StructField("official", T.BooleanType(), True),
                T.StructField("published_at", T.StringType(), True),
                T.StructField("site", T.StringType(), True),
                T.StructField("size", T.LongType(), True),
                T.StructField("type", T.StringType(), True),
            ])
        ), True)
    ]), True),
])

In [5]:
input_path = "/home/dphung/projects/tmdb/data/raw/movie_details/snapshot_date=2026-04-05/movie_id=467914.json"


In [ ]:
df = spark.read.option("multiline", "true").json(input_path)

# Optional: inspect schema
df.printSchema()

In [ ]:
# 3. Add snapshot_date
# ---------------------------
# For now, hard-code it. Later you can extract it from folder name like snapshot_date=2026-04-05
snapshot_date = "2026-04-05"

df = df.withColumn("snapshot_date", F.lit(snapshot_date).cast("date"))


In [9]:
# 4. Core movie table
# ---------------------------
movies = (
    df.select(
        F.col("id").alias("movie_id"),
        "title",
        "original_title",
        "original_language",
        "overview",
        "homepage",
        "imdb_id",
        "release_date",
        "runtime",
        "status",
        "tagline",
        "adult",
        "video",
        "budget",
        "revenue",
        "popularity",
        "vote_average",
        "vote_count",
        "poster_path",
        "backdrop_path",
        "snapshot_date"
    )
)
movies.show()

+--------+--------------------+--------------------+-----------------+--------------------+--------------------+---------+------------+-------+--------+--------------------+-----+-----+------+-------+----------+------------+----------+--------------------+--------------------+-------------+
|movie_id|               title|      original_title|original_language|            overview|            homepage|  imdb_id|release_date|runtime|  status|             tagline|adult|video|budget|revenue|popularity|vote_average|vote_count|         poster_path|       backdrop_path|snapshot_date|
+--------+--------------------+--------------------+-----------------+--------------------+--------------------+---------+------------+-------+--------+--------------------+-----+-----+------+-------+----------+------------+----------+--------------------+--------------------+-------------+
|  467914|The Land of Somet...|The Land of Somet...|               en|Twins Alfie and E...|https://www.thela...|tt3517106|  